# 1. Data Loading and Preprocessing

This section handles loading the Fashion-MNIST dataset and splitting it into training, validation, and test sets.

## 1.0. Utility Functions

In [34]:
import numpy as np
import random
import os
# import tensorflow as tf
import torch

SEED = 42

def set_all_seeds(seed):
    """Sets seeds for reproducibility across common libraries."""
    # Standard Python
    random.seed(seed)
    
    # NumPy
    np.random.seed(seed)
    
    # # TensorFlow/Keras
    # tf.random.set_seed(seed)
    
    # PyTorch
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    
    # OS environment
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f"Seeds set to {seed} successfully!")

# Call the function at the very beginning of your notebook
set_all_seeds(SEED)

Seeds set to 42 successfully!


In [35]:
def seed_worker(worker_id):
    """Sets a unique seed for each DataLoader worker."""
    worker_seed = SEED + worker_id # Ensure a unique seed per worker
    np.random.seed(worker_seed)
    random.seed(worker_seed)
    
g = torch.Generator()
g.manual_seed(SEED)

In [36]:
# ============================================================================
# CONFIGURATION AND UTILITY FUNCTIONS
# ============================================================================

import matplotlib.pyplot as plt

# Student username for plot labeling (as per submission requirements)
USERNAME = "bhogadi.likhit"

def add_username_to_plot(ax=None, username=USERNAME):
    """
    Add username watermark to plot as per submission requirements.
    
    Args:
        ax: Matplotlib axis object. If None, uses current axis.
        username: Username string to display
    """
    if ax is None:
        ax = plt.gca()
    
    plt.text(
        0.95, 0.95, username,
        ha='right', va='top',
        transform=ax.transAxes,
        fontsize=10, color='gray', alpha=0.7
    )

# print(f"✓ Configuration loaded")
# print(f"✓ Username: {USERNAME}")

## 1.1. Dataset Loading Function

Load Fashion-MNIST and split into train (90%), validation (10%), and test sets.

In [37]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import random_split

# For type hinting, which is good practice
from torch.utils.data import Dataset
from typing import Tuple

def load_fashion_data() -> Tuple[Dataset, Dataset, Dataset]:
    """
    Loads the Fashion-MNIST dataset and splits the data into train, val, test.

    This function loads the Fashion-MNIST training and test sets from
    torchvision. It then splits the original training set into a new,
    smaller training set and a validation set (90/10 ratio).

    Returns:
        Tuple[Dataset, Dataset, Dataset]: A tuple containing the training,
        validation, and test datasets as torch.utils.data.Dataset objects.
    """
    # Use a basic transform to convert images to tensors.
    # More complex transforms will be applied later in the custom Dataset class.
    # converts a PIL Image or numpy.ndarray to tensor.
    transform = transforms.ToTensor()

    # 1. Load the full original training data
    full_train_dataset = datasets.FashionMNIST(
        root='./data',
        train=True,
        download=True,
        transform=transform
    )

    # 2. Load the test data
    test_dataset = datasets.FashionMNIST(
        root='./data',
        train=False,
        download=True,
        transform=transform
    )

    # 3. Calculate split sizes
    dataset_size = len(full_train_dataset)
    train_size = int(0.9 * dataset_size)
    val_size = dataset_size - train_size

    # 4. Split the full training dataset into training and validation subsets
    # Using a fixed generator seed for reproducibility of the split
    generator = torch.Generator()
    train_dataset, val_dataset = random_split(
        full_train_dataset,
        [train_size, val_size],
        generator=generator
    )

    return train_dataset, val_dataset, test_dataset

# # Example of how to call the function and verify the output
# if __name__ == '__main__':
#     train_data, val_data, test_data = load_fashion_data()

#     print(f"Fashion-MNIST Dataset Summary:")
#     print(f"  - Total Original Training Images: {len(train_data) + len(val_data)}")
#     print(f"  - Split Training Set Size:      {len(train_data)}")
#     print(f"  - Split Validation Set Size:    {len(val_data)}")
#     print(f"  - Test Set Size:                {len(test_data)}")

## 1.1a. Mean and Standard Deviation Calculation

Calculate dataset statistics for normalization (instead of using pre-computed values).

In [38]:
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from typing import Tuple

def calculate_mean_std(dataset: Dataset) -> Tuple[Tuple[float, ...], Tuple[float, ...]]:
    """
    Calculate the mean and standard deviation of a dataset.
    
    Args:
        dataset: A PyTorch dataset containing images
        
    Returns:
        Tuple containing mean and std as tuples
    """
    # Stack all images into a single tensor
    # We'll iterate through the dataset and collect all pixel values
    all_pixels = []
    
    for img, _ in dataset:
        # img is already a tensor from ToTensor() transform
        all_pixels.append(img)
    
    # Stack all images: shape will be [N, C, H, W]
    all_pixels = torch.stack(all_pixels)
    
    # Calculate mean and std across all dimensions except the channel dimension
    # Compute per-channel statistics
    mean = all_pixels.mean(dim=[0, 2, 3])  # Average over batch, height, width
    std = all_pixels.std(dim=[0, 2, 3])    # Std dev over batch, height, width
    
    return tuple(mean.tolist()), tuple(std.tolist())

# # Load the training data to calculate statistics
# train_dataset, val_dataset, test_dataset = load_fashion_data()

# # Calculate mean and std from the training set
# print("Calculating mean and standard deviation from training data...")
# FASHION_MNIST_MEAN, FASHION_MNIST_STD = calculate_mean_std(train_dataset)

# print(f"Calculated Mean: {FASHION_MNIST_MEAN}")
# print(f"Calculated Std:  {FASHION_MNIST_STD}")

## 1.2. Custom Datatset class:

In [39]:
class FashionMNISTDataset(Dataset):
    """
    Custom Dataset wrapper for Fashion-MNIST that returns:
    - Normalized image tensor
    - Class label
    - Ink target (normalized pixel intensity)
    
    Supports optional data augmentation for training.
    """
    
    def __init__(
        self, 
        base_dataset: Dataset,
        mean: Tuple[float, ...],
        std: Tuple[float, ...],
        augment: bool = False
    ):
        """
        Initialize the custom Fashion-MNIST dataset.
        
        Args:
            base_dataset: The underlying Fashion-MNIST dataset (can be a Subset)
            mean: Mean values for normalization (per channel)
            std: Standard deviation values for normalization (per channel)
            augment: Whether to apply data augmentation (for training only)
        """
        self.base_dataset = base_dataset
        self.mean = mean
        self.std = std
        self.augment = augment
        
        # Create normalization transform
        self.normalize = transforms.Normalize(mean=mean, std=std)
        
        # Create augmentation transforms (light augmentations)
        if self.augment:
            self.augmentation = transforms.Compose([
                # Random crop: crop to 28x28 from a 32x32 padded image
                # This creates slight translations
                transforms.RandomCrop(28, padding=4),
                
                # Small random rotation (up to 10 degrees)
                transforms.RandomRotation(degrees=10),
                
                # Random horizontal flip (50% chance)
                # Note: This is reasonable for most Fashion-MNIST items
                transforms.RandomHorizontalFlip(p=0.5),
            ])
        
    def __len__(self) -> int:
        """Return the total number of samples in the dataset."""
        return len(self.base_dataset)
    
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int, float]:
        """
        Get a single item from the dataset.
        
        Args:
            idx: Index of the item to retrieve
            
        Returns:
            Tuple containing:
                - image: Normalized (and optionally augmented) image tensor of shape [C, H, W]
                - class_label: Integer label in range [0, 9]
                - ink_target: Normalized pixel intensity (average pixel value)
        """
        # Get the original image and label from the base dataset
        image, class_label = self.base_dataset[idx]
        
        # Calculate ink target BEFORE any transformations
        # Ink is the average pixel value (mean intensity) of the original image
        # image is already a tensor from ToTensor() with values in [0, 1]
        ink_target = float(image.mean().item())
        
        # Apply augmentation if enabled (training set only)
        if self.augment:
            image = self.augmentation(image)
        
        # Normalize the image using the dataset mean and std
        normalized_image = self.normalize(image)
        
        return normalized_image, class_label, ink_target

## 1.3. Data Augmentation

We apply light augmentations to the training set only to improve generalization:
- **Random Crop**: Crop 28x28 from a 32x32 padded image (creates small translations)
- **Random Rotation**: Small rotations up to ±10 degrees
- **Random Horizontal Flip**: 50% chance of horizontal flip

Validation and test sets remain unaltered for fair evaluation.

(Done in the Custom Dataset class)

# 2. Model Implementation: Multi-Task CNN

## 2.1. Architecture, 2.2. Forward Pass

In [40]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import List, Tuple

class MultiTaskCNN(nn.Module):
    """
    Configurable Multi-Task CNN for Fashion-MNIST classification and ink regression.
    
    Architecture:
    - Flexible convolutional backbone with configurable number of layers and filters
    - Two separate heads:
        1. Classification head: outputs logits [B, 10] for 10 classes
        2. Regression head: outputs scalar values [B, 1] for ink prediction
    """
    
    def __init__(
        self, 
        conv_channels: List[int] = [32, 64, 128],
        fc_class_hidden: int = 256,
        fc_regress_hidden: int = 128,
        dropout_rate: float = 0.5,
        input_size: Tuple[int, int] = (28, 28)
    ):
        """
        Initialize the Multi-Task CNN with configurable architecture.
        
        Args:
            conv_channels: List of output channels for each conv layer (default: [32, 64, 128])
                          Length determines number of conv blocks
            fc_class_hidden: Hidden layer size for classification head (default: 256)
            fc_regress_hidden: Hidden layer size for regression head (default: 128)
            dropout_rate: Dropout probability for regularization (default: 0.5)
            input_size: Input image size (H, W) (default: (28, 28))
        """
        super(MultiTaskCNN, self).__init__()
        
        self.conv_channels = conv_channels
        self.num_conv_layers = len(conv_channels)
        self.dropout_rate = dropout_rate
        self.input_size = input_size
        
        # =====================================================================
        # SHARED CONVOLUTIONAL BACKBONE (DYNAMICALLY CONSTRUCTED)
        # =====================================================================
        
        # Build convolutional layers dynamically
        self.conv_layers = nn.ModuleList()
        self.bn_layers = nn.ModuleList()
        self.pool_layers = nn.ModuleList()
        
        in_channels = 1  # Grayscale input
        for i, out_channels in enumerate(conv_channels):
            # Convolutional layer
            self.conv_layers.append(
                nn.Conv2d(
                    in_channels=in_channels,
                    out_channels=out_channels,
                    kernel_size=3,
                    padding=1
                )
            )
            # Batch normalization
            self.bn_layers.append(nn.BatchNorm2d(out_channels))
            # Max pooling (2x2)
            self.pool_layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
            
            in_channels = out_channels  # Update for next layer
        
        # Dropout for regularization
        self.dropout = nn.Dropout(p=dropout_rate)
        
        # Calculate the size of flattened features after all pooling layers
        # Each pooling layer halves the spatial dimensions
        h, w = input_size
        for _ in range(self.num_conv_layers):
            h = h // 2
            w = w // 2
        
        self.feature_size = conv_channels[-1] * h * w
        
        # =====================================================================
        # CLASSIFICATION HEAD (for class labels)
        # =====================================================================
        self.fc_class1 = nn.Linear(self.feature_size, fc_class_hidden)
        self.bn_class1 = nn.BatchNorm1d(fc_class_hidden)
        self.fc_class2 = nn.Linear(fc_class_hidden, 10)  # 10 classes for Fashion-MNIST
        
        # =====================================================================
        # REGRESSION HEAD (for ink prediction)
        # =====================================================================
        self.fc_regress1 = nn.Linear(self.feature_size, fc_regress_hidden)
        self.bn_regress1 = nn.BatchNorm1d(fc_regress_hidden)
        self.fc_regress2 = nn.Linear(fc_regress_hidden, 1)  # Single scalar output
        
    def forward_shared(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass through the shared convolutional backbone.
        
        Args:
            x: Input tensor of shape [B, 1, H, W]
            
        Returns:
            Flattened features of shape [B, feature_size]
        """
        # Pass through all convolutional blocks dynamically
        for i in range(self.num_conv_layers):
            x = self.conv_layers[i](x)
            x = self.bn_layers[i](x)
            x = F.relu(x)
            x = self.pool_layers[i](x)
        
        # Flatten the feature maps
        x = x.view(x.size(0), -1)
        
        # Apply dropout
        x = self.dropout(x)
        
        return x
    
    def forward(
        self, 
        x: torch.Tensor, 
        return_features: bool = False
    ) -> tuple[torch.Tensor, torch.Tensor] | tuple[torch.Tensor, torch.Tensor, dict]:
        """
        Forward pass through the entire network.
        
        Args:
            x: Input tensor of shape [B, 1, H, W]
            return_features: If True, also return intermediate feature maps for visualization
            
        Returns:
            If return_features=False:
                Tuple containing:
                    - class_logits: Classification logits of shape [B, 10]
                    - ink_pred: Ink predictions of shape [B, 1]
            
            If return_features=True:
                Tuple containing:
                    - class_logits: Classification logits of shape [B, 10]
                    - ink_pred: Ink predictions of shape [B, 1]
                    - features: Dictionary with intermediate feature maps
        """
        # Dictionary to store intermediate features if requested
        features = {} if return_features else None
        
        # ===== Shared Convolutional Backbone =====
        
        # Pass through all convolutional blocks dynamically
        for i in range(self.num_conv_layers):
            x = self.conv_layers[i](x)
            x = self.bn_layers[i](x)
            x = F.relu(x)
            x = self.pool_layers[i](x)
            
            if return_features:
                features[f'conv{i+1}'] = x.clone()
        
        # Flatten the feature maps
        x = x.view(x.size(0), -1)
        
        # Apply dropout
        shared_features = self.dropout(x)
        if return_features:
            features['shared_features'] = shared_features.clone()
        
        # ===== Classification Head =====
        x_class = self.fc_class1(shared_features)
        x_class = self.bn_class1(x_class)
        x_class = F.relu(x_class)
        x_class = self.dropout(x_class)
        class_logits = self.fc_class2(x_class)  # [B, 10]
        
        # ===== Regression Head =====
        x_regress = self.fc_regress1(shared_features)
        x_regress = self.bn_regress1(x_regress)
        x_regress = F.relu(x_regress)
        x_regress = self.dropout(x_regress)
        ink_pred = self.fc_regress2(x_regress)  # [B, 1]
        
        # Return with or without features
        if return_features:
            return class_logits, ink_pred, features
        else:
            return class_logits, ink_pred
    
    def get_num_parameters(self) -> int:
        """
        Calculate the total number of trainable parameters in the model.
        
        Returns:
            Total number of trainable parameters
        """
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

## 2.3 Loss Functions

This section implements the joint loss function for multi-task learning

In [41]:
# ============================================================================
# JOINT LOSS FUNCTION FOR MULTI-TASK LEARNING
# ============================================================================

import torch
import torch.nn as nn
from typing import Tuple

class MultiTaskLoss(nn.Module):
    """
    Joint loss function for multi-task learning.
    
    Combines classification loss (Cross-Entropy) and regression loss (MSE)
    with configurable weights.
    
    L_total = lambda1 * L_CE + lambda2 * L_MSE
    """
    
    def __init__(self, lambda1: float = 1.0, lambda2: float = 1.0):
        """
        Initialize the multi-task loss function.
        
        Args:
            lambda1: Weight for classification loss (default: 1.0)
            lambda2: Weight for regression loss (default: 1.0)
        """
        super(MultiTaskLoss, self).__init__()
        
        self.lambda1 = lambda1
        self.lambda2 = lambda2
        
        # Define individual loss functions
        self.classification_loss = nn.CrossEntropyLoss()  # For class labels
        self.regression_loss = nn.MSELoss()  # For ink prediction
        
    def forward(
        self, 
        class_logits: torch.Tensor, 
        class_targets: torch.Tensor,
        ink_pred: torch.Tensor, 
        ink_targets: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Compute the joint loss.
        
        Args:
            class_logits: Predicted class logits [B, 10]
            class_targets: True class labels [B] (integer labels 0-9)
            ink_pred: Predicted ink values [B, 1]
            ink_targets: True ink values [B] (float values)
            
        Returns:
            Tuple containing:
                - total_loss: Weighted sum of both losses
                - ce_loss: Classification loss value
                - mse_loss: Regression loss value
        """
        # Compute Cross-Entropy loss for classification
        ce_loss = self.classification_loss(class_logits, class_targets)
        
        # Reshape tensors for MSE loss if needed
        # ink_pred is [B, 1], ink_targets should be [B, 1]
        if ink_targets.dim() == 1:
            ink_targets = ink_targets.unsqueeze(1)  # [B] -> [B, 1]
        
        # Compute MSE loss for regression
        mse_loss = self.regression_loss(ink_pred, ink_targets)
        
        # Compute weighted total loss
        total_loss = self.lambda1 * ce_loss + self.lambda2 * mse_loss
        
        return total_loss, ce_loss, mse_loss
    
    def __repr__(self) -> str:
        return (f"MultiTaskLoss(lambda1={self.lambda1}, lambda2={self.lambda2})\n"
                f"  - Classification: CrossEntropyLoss (weight={self.lambda1})\n"
                f"  - Regression: MSELoss (weight={self.lambda2})")

# print("✓ MultiTaskLoss class defined successfully")

# 3. Hyperparameter Tuning and wandb Logging

This section implements comprehensive hyperparameter tuning with W&B logging to track experiments and compare different configurations.

## 3.1 Install and Setup Weights & Biases

First, we need to install wandb and set up authentication.

In [42]:
# ============================================================================
# INSTALL AND SETUP WANDB
# ============================================================================

# Install wandb if not already installed
# !pip install wandb -q

import wandb
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np
import os
import dotenv

# Load environment variables from .env file
dotenv.load_dotenv()

# Get the API key from the environment variable
api_key = os.getenv("WANDB_API_KEY")

# Login to wandb (you'll need to provide your API key)
wandb.login(key=api_key)

print("✓ wandb imported successfully")
print("✓ To login, run: wandb.login() and enter your API key")
print("✓ Get your API key from: https://wandb.ai/authorize")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home2/bhogadi.likhit/.netrc


✓ wandb imported successfully
✓ To login, run: wandb.login() and enter your API key
✓ Get your API key from: https://wandb.ai/authorize


Define multiple configurations to explore different hyperparameters and loss weights.

## 3.2 Prepare Data

In [43]:
# ============================================================================
# PREPARE DATA LOADERS
# ============================================================================

# First, load and prepare datasets (if not already done)
# Uncomment these if running from scratch

# Load data
train_dataset, val_dataset, test_dataset = load_fashion_data()

# Calculate mean and std
FASHION_MNIST_MEAN, FASHION_MNIST_STD = calculate_mean_std(train_dataset)

# Create custom datasets with augmentation
train_custom = FashionMNISTDataset(train_dataset, FASHION_MNIST_MEAN, FASHION_MNIST_STD, augment=True)
val_custom = FashionMNISTDataset(val_dataset, FASHION_MNIST_MEAN, FASHION_MNIST_STD, augment=False)
test_custom = FashionMNISTDataset(test_dataset, FASHION_MNIST_MEAN, FASHION_MNIST_STD, augment=False)

def create_dataloaders(batch_size):
    """Create data loaders with specified batch size."""
    train_loader = DataLoader(
        train_custom,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        worker_init_fn=seed_worker,
        generator=g
    )
    
    val_loader = DataLoader(
        val_custom,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
        worker_init_fn=seed_worker,
        generator=g
    )
    
    test_loader = DataLoader(
        test_custom,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
        worker_init_fn=seed_worker,
        generator=g
    )
    
    return train_loader, val_loader, test_loader

print("✓ Data loader function defined")
print("⚠️  Remember to uncomment data loading code above before running experiments!")

✓ Data loader function defined
⚠️  Remember to uncomment data loading code above before running experiments!


## 3.3 Training and Evaluation Functions

Implement comprehensive training loop with metric tracking.

In [44]:
# ============================================================================
# TRAINING FUNCTION
# ============================================================================

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    """
    Train the model for one epoch.
    
    Args:
        model: MultiTaskCNN model
        dataloader: Training data loader
        criterion: MultiTaskLoss function
        optimizer: Optimizer
        device: Device to train on
        
    Returns:
        Dictionary with average losses and metrics
    """
    model.train()
    
    total_loss_sum = 0.0
    ce_loss_sum = 0.0
    mse_loss_sum = 0.0
    correct = 0
    total = 0
    
    for images, class_labels, ink_targets in tqdm(dataloader, desc="Training", leave=False):
        # Move data to device
        images = images.to(device)
        class_labels = class_labels.to(device)
        ink_targets = ink_targets.to(device).float()
        
        # Zero gradients
        optimizer.zero_grad()
        
        # Forward pass
        class_logits, ink_pred = model(images)
        
        # Compute loss
        total_loss, ce_loss, mse_loss = criterion(
            class_logits, class_labels,
            ink_pred, ink_targets
        )
        
        # Backward pass
        total_loss.backward()
        optimizer.step()
        
        # Accumulate losses
        total_loss_sum += total_loss.item()
        ce_loss_sum += ce_loss.item()
        mse_loss_sum += mse_loss.item()
        
        # Calculate accuracy
        _, predicted = torch.max(class_logits, 1)
        total += class_labels.size(0)
        correct += (predicted == class_labels).sum().item()
    
    # Calculate averages
    num_batches = len(dataloader)
    metrics = {
        'train_loss': total_loss_sum / num_batches,
        'train_ce_loss': ce_loss_sum / num_batches,
        'train_mse_loss': mse_loss_sum / num_batches,
        'train_accuracy': 100.0 * correct / total
    }
    
    return metrics # per batch it is 

print("✓ Training function defined")

✓ Training function defined


In [45]:
# ============================================================================
# EVALUATION FUNCTION
# ============================================================================

def evaluate(model, dataloader, criterion, device, phase='Validation'):
    """
    Evaluate the model.
    
    Args:
        model: MultiTaskCNN model
        dataloader: Validation/Test data loader
        criterion: MultiTaskLoss function
        device: Device to evaluate on
        phase: 'Validation' or 'Test'
        
    Returns:
        Dictionary with losses and metrics
    """
    model.eval()
    
    total_loss_sum = 0.0
    ce_loss_sum = 0.0
    mse_loss_sum = 0.0
    correct = 0
    total = 0
    
    all_ink_preds = []
    all_ink_targets = []
    
    with torch.no_grad():
        for images, class_labels, ink_targets in tqdm(dataloader, desc=f"{phase}", leave=False):
            # Move data to device
            images = images.to(device)
            class_labels = class_labels.to(device)
            ink_targets = ink_targets.to(device).float()
            
            # Forward pass
            class_logits, ink_pred = model(images)
            
            # Compute loss
            total_loss, ce_loss, mse_loss = criterion(
                class_logits, class_labels,
                ink_pred, ink_targets
            )
            
            # Accumulate losses
            total_loss_sum += total_loss.item()
            ce_loss_sum += ce_loss.item()
            mse_loss_sum += mse_loss.item()
            
            # Calculate accuracy
            _, predicted = torch.max(class_logits, 1)
            total += class_labels.size(0)
            correct += (predicted == class_labels).sum().item()
            
            # Collect predictions for regression metrics
            all_ink_preds.append(ink_pred.cpu())
            all_ink_targets.append(ink_targets.cpu().unsqueeze(1) if ink_targets.dim() == 1 else ink_targets.cpu())
    
    # Calculate averages
    num_batches = len(dataloader)
    
    # Concatenate all predictions
    all_ink_preds = torch.cat(all_ink_preds)
    all_ink_targets = torch.cat(all_ink_targets)
    
    # Calculate regression metrics
    mae = torch.mean(torch.abs(all_ink_preds - all_ink_targets)).item()
    rmse = torch.sqrt(torch.mean((all_ink_preds - all_ink_targets) ** 2)).item()
    
    prefix = phase.lower()
    metrics = {
        f'{prefix}_loss': total_loss_sum / num_batches,
        f'{prefix}_ce_loss': ce_loss_sum / num_batches,
        f'{prefix}_mse_loss': mse_loss_sum / num_batches,
        f'{prefix}_accuracy': 100.0 * correct / total,
        f'{prefix}_mae': mae,
        f'{prefix}_rmse': rmse
    }
    
    return metrics

print("✓ Evaluation function defined")

✓ Evaluation function defined


In [46]:
# ============================================================================
# MAIN TRAINING FUNCTION WITH WANDB LOGGING
# ============================================================================

def train_model_with_wandb(config, train_loader, val_loader, run_name, device='cuda'):
    """
    Train model with full wandb logging.
    
    NOTE: This function does NOT use the test set during training!
    Test set should only be used ONCE at the very end for final evaluation.
    
    Args:
        config: Configuration dictionary
        train_loader: Training data loader
        val_loader: Validation data loader
        run_name: Name for this wandb run
        device: Device to train on
        
    Returns:
        Trained model and metrics history
    """
    # Initialize wandb
    wandb.init(
        project="fashion-mnist-multitask",
        name=run_name,
        config=config,
        tags=[
            config['optimizer'], 
            f"lambda1={config['lambda1']}", 
            f"lambda2={config['lambda2']}",
            f"layers={len(config['conv_channels'])}"
        ]
    )
    
    # Create model with configurable architecture
    model = MultiTaskCNN(
        conv_channels=config['conv_channels'],
        fc_class_hidden=config['fc_class_hidden'],
        fc_regress_hidden=config['fc_regress_hidden'],
        dropout_rate=config['dropout_rate']
    ).to(device)
    
    # Log model size
    num_params = model.get_num_parameters()
    print(f"Model architecture: {len(config['conv_channels'])} conv layers {config['conv_channels']}")
    print(f"Total parameters: {num_params:,}")
    
    # Create loss function
    criterion = MultiTaskLoss(lambda1=config['lambda1'], lambda2=config['lambda2'])
    
    # Create optimizer
    if config['optimizer'] == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=config['learning_rate'])
    elif config['optimizer'] == 'AdamW':
        optimizer = optim.AdamW(model.parameters(), lr=config['learning_rate'])
    elif config['optimizer'] == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=config['learning_rate'], momentum=0.9)
    else:
        raise ValueError(f"Unknown optimizer: {config['optimizer']}")
    
    # Log model architecture
    wandb.watch(model, log='all', log_freq=100)
    
    # Training loop - ONLY use train and validation sets
    best_val_accuracy = 0.0
    best_val_rmse = float('inf')
    history = {
        'train_loss': [], 'val_loss': [],
        'train_accuracy': [], 'val_accuracy': [],
        'val_rmse': []
    }
    
    print(f"\n{'='*80}")
    print(f"Starting Training: {run_name}")
    print(f"{'='*80}")
    
    for epoch in range(config['epochs']):
        print(f"\nEpoch {epoch+1}/{config['epochs']}")
        
        # Train
        train_metrics = train_one_epoch(model, train_loader, criterion, optimizer, device)
        
        # Validate (NOT TEST!)
        val_metrics = evaluate(model, val_loader, criterion, device, phase='Validation')
        
        # Log to wandb
        wandb.log({
            'epoch': epoch + 1,
            **train_metrics,
            **val_metrics
        })
        
        # Print epoch summary
        print(f"  Train Loss: {train_metrics['train_loss']:.4f} | "
              f"Train Acc: {train_metrics['train_accuracy']:.2f}%")
        print(f"  Val Loss: {val_metrics['validation_loss']:.4f} | "
              f"Val Acc: {val_metrics['validation_accuracy']:.2f}% | "
              f"Val RMSE: {val_metrics['validation_rmse']:.4f}")
        
        # Save history
        history['train_loss'].append(train_metrics['train_loss'])
        history['val_loss'].append(val_metrics['validation_loss'])
        history['train_accuracy'].append(train_metrics['train_accuracy'])
        history['val_accuracy'].append(val_metrics['validation_accuracy'])
        history['val_rmse'].append(val_metrics['validation_rmse'])
        
        # Track best models based on VALIDATION metrics only
        if val_metrics['validation_accuracy'] > best_val_accuracy:
            best_val_accuracy = val_metrics['validation_accuracy']
            wandb.run.summary['best_val_accuracy'] = best_val_accuracy
            
        if val_metrics['validation_rmse'] < best_val_rmse:
            best_val_rmse = val_metrics['validation_rmse']
            wandb.run.summary['best_val_rmse'] = best_val_rmse
    
    print(f"\n{'='*80}")
    print("Training Complete!")
    print(f"Best Validation Accuracy: {best_val_accuracy:.2f}%")
    print(f"Best Validation RMSE: {best_val_rmse:.4f}")
    print(f"{'='*80}")
    
    # Close wandb run
    wandb.finish()
    
    return model, history

print("✓ Main training function with wandb logging defined")

✓ Main training function with wandb logging defined


Prepare data loaders and execute all training runs.

## 3.4 Hyperparameter Configurations

In [47]:
# ============================================================================
# HYPERPARAMETER CONFIGURATIONS FOR EXPERIMENTS
# ============================================================================

# Define at least 5 different configurations to experiment with
# Now with configurable architecture parameters!
experiment_configs = {
    'run_1_baseline': {
        'learning_rate': 0.001,
        'dropout_rate': 0.5,
        'optimizer': 'AdamW',
        'batch_size': 64,
        'lambda1': 1.0,
        'lambda2': 1.0,
        'conv_channels': [32, 64, 128],  # 3 layers: 32->64->128
        'fc_class_hidden': 256,
        'fc_regress_hidden': 128,
        'epochs': 20,
        'description': 'Baseline: 3 conv layers [32,64,128], standard hyperparameters'
    },
}

# Display all configurations
print("="*80)
print("EXPERIMENT CONFIGURATIONS")
print("="*80)

for run_name, config in experiment_configs.items():
    print(f"\n{run_name.upper()}")
    print(f"  Description: {config['description']}")
    print(f"  Architecture:")
    print(f"    Conv Channels: {config['conv_channels']} ({len(config['conv_channels'])} layers)")
    print(f"    FC Class Hidden: {config['fc_class_hidden']}")
    print(f"    FC Regress Hidden: {config['fc_regress_hidden']}")
    print(f"  Training:")
    print(f"    Learning Rate: {config['learning_rate']}")
    print(f"    Dropout: {config['dropout_rate']}")
    print(f"    Optimizer: {config['optimizer']}")
    print(f"    Batch Size: {config['batch_size']}")
    print(f"    λ₁ (Classification): {config['lambda1']}")
    print(f"    λ₂ (Regression): {config['lambda2']}")
    print(f"    Epochs: {config['epochs']}")

print("\n" + "="*80)
print(f"Total configurations: {len(experiment_configs)}")
print("="*80)

EXPERIMENT CONFIGURATIONS

RUN_1_BASELINE
  Description: Baseline: 3 conv layers [32,64,128], standard hyperparameters
  Architecture:
    Conv Channels: [32, 64, 128] (3 layers)
    FC Class Hidden: 256
    FC Regress Hidden: 128
  Training:
    Learning Rate: 0.001
    Dropout: 0.5
    Optimizer: AdamW
    Batch Size: 64
    λ₁ (Classification): 1.0
    λ₂ (Regression): 1.0
    Epochs: 20

Total configurations: 1


## 3.5 Run Experiments

In [48]:
# ============================================================================
# RUN ALL EXPERIMENTS
# ============================================================================

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Store results from all runs
all_results = {}

# Run all experiments (ONLY using train and validation sets)
for run_name, config in experiment_configs.items():
    print(f"\n\n{'#'*80}")
    print(f"# Starting: {run_name}")
    print(f"{'#'*80}\n")
    
    # Create data loaders with config-specific batch size
    train_loader, val_loader, _ = create_dataloaders(config['batch_size'])
    
    # Train model (NO TEST SET USED HERE!)
    model, history = train_model_with_wandb(
        config=config,
        train_loader=train_loader,
        val_loader=val_loader,
        run_name=run_name,
        device=device
    )
    
    # Store results (validation metrics only during training)
    all_results[run_name] = {
        'config': config,
        'history': history,
        'model': model,  # Save model for later test evaluation
        'best_val_accuracy': max(history['val_accuracy']),
        'best_val_rmse': min(history['val_rmse'])
    }
    
    print(f"\n✓ Completed: {run_name}\n")

print(f"\n\n{'='*80}")
print("ALL EXPERIMENTS COMPLETED!")
print(f"{'='*80}")
print(f"Total runs: {len(all_results)}")
print("Check wandb dashboard for detailed logs and visualizations")
print(f"{'='*80}")

Using device: cuda


################################################################################
# Starting: run_1_baseline
################################################################################



Model architecture: 3 conv layers [32, 64, 128]
Total parameters: 539,339

Starting Training: run_1_baseline

Epoch 1/20


  Train Loss: 0.7783 | Train Acc: 72.10%
  Val Loss: 0.4970 | Val Acc: 81.48% | Val RMSE: 0.0747

Epoch 2/20


  Train Loss: 0.5585 | Train Acc: 79.29%
  Val Loss: 0.4103 | Val Acc: 84.22% | Val RMSE: 0.0306

Epoch 3/20


  Train Loss: 0.5014 | Train Acc: 81.63%
  Val Loss: 0.3704 | Val Acc: 86.27% | Val RMSE: 0.0355

Epoch 4/20


  Train Loss: 0.4651 | Train Acc: 83.15%
  Val Loss: 0.3521 | Val Acc: 86.45% | Val RMSE: 0.0256

Epoch 5/20


... (15 outputs left out)

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


  Train Loss: 0.3392 | Train Acc: 87.65%
  Val Loss: 0.2498 | Val Acc: 91.03% | Val RMSE: 0.0342

Training Complete!
Best Validation Accuracy: 91.03%
Best Validation RMSE: 0.0256


epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_accuracy,▁▄▅▆▆▆▇▇▇▇▇█████████
train_ce_loss,█▅▄▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁
train_loss,█▄▄▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁
train_mse_loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
validation_accuracy,▁▃▅▅▆▅▆▆▆▆▆▇█▇█▇████
validation_ce_loss,█▆▄▄▃▃▃▂▂▃▂▁▁▂▁▂▁▁▁▁
validation_loss,█▆▄▄▃▃▃▂▂▃▂▁▁▂▁▂▁▁▁▁
validation_mae,█▂▂▁▁▂▂▁▂▂▂▁▁▁▂▃▂▂▂▂
validation_mse_loss,█▁▂▁▁▂▁▁▁▂▁▁▁▁▂▂▁▁▁▂
+1,...



✓ Completed: run_1_baseline



ALL EXPERIMENTS COMPLETED!
Total runs: 1
Check wandb dashboard for detailed logs and visualizations
